# Deploy IQ spoke

**Run this notebook once before the rest of the Foundry IQ series.**

Deploys the Foundry IQ resources into the **existing** `rg-foundry-multi-{suffix}`
resource group, extending the shared AI Foundry account from the multi-project deployment with a new
`iq-project` and a dedicated Azure AI Search service.

## What gets deployed

| Resource | Type | Where |
|----------|------|-------|
| `iq-search-{suffix}` | Azure AI Search (Basic, SystemAssigned) | `rg-foundry-multi-{suffix}` - new |
| `iq-project` | Foundry Project | Child of existing `aif-spoke-multi-{suffix}` account - new |
| `iq-apim-connection` | Project connection (ApiManagement) | On `iq-project` - new |
| RBAC assignments | Deployer + project MI + search MI | On AI Account + Search service |

> **No new resource group or Foundry account is created.** The IQ lab reuses
> the shared account from the multi-project deployment, demonstrating the 1:N multi-project pattern
> growing to absorb a new workload capability.

## Sequence

```
10-01-deploy-search-and-project ← this notebook (Bicep deployment, .env outputs)
10-02-index-and-ingest          ← creates the arxiv-nlp search index + uploads 3,000 documents
10-04-search-patterns           ← demonstrates 6 retrieval patterns
10-03-knowledge-base-setup      ← knowledge source, knowledge bases, MCP connection, agent
10-05-agent-iq-queries          ← agent queries via Responses API
```

## Prerequisites

1. **Multi-project deployed** - `05-04-01-deploy-foundry-multi-project.ipynb` must have run successfully.
   The `.env` file must contain `MULTI_ACCOUNT`, `GATEWAY_URL`, `ALPHA_GATEWAY_KEY`,
   `CHAT_MODEL`, and `EMBEDDING_MODEL`.
2. **Python environment** - run `uv sync` from the repo root; select the `.venv` kernel.
3. **Azure CLI** - run `az login` before executing cells.

## Step 1: Load configuration

In [1]:
import os
import json
import subprocess
import base64
import time
import tempfile
from pathlib import Path

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
env_file = repo_root / '.env'

# Read .env manually: same pattern as other deploy notebooks
with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

GATEWAY_URL    = os.environ['GATEWAY_URL']
CHAT_MODEL     = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')
EMBED_MODEL    = os.environ.get('EMBEDDING_MODEL', 'text-embedding-3-large')
MULTI_ACCOUNT  = os.environ['MULTI_ACCOUNT']   # aif-spoke-multi-{suffix}, from the multi-project deployment

# Bootstrap key: used for the initial Bicep deployment.
# Step 5 creates a dedicated IQ subscription and patches the connection.
BOOTSTRAP_KEY  = os.environ['ALPHA_GATEWAY_KEY']

print(f'Gateway URL   : {GATEWAY_URL}')
print(f'Chat model    : {CHAT_MODEL}')
print(f'Multi account : {MULTI_ACCOUNT}')
print(f'Bootstrap key : {BOOTSTRAP_KEY[:4]}... (hidden)')

Gateway URL   : https://apim-foundry-c2676f.azure-api.net/openai
Chat model    : gpt-4.1-mini
Multi account : aif-spoke-multi-c2676f
Bootstrap key : 8382... (hidden)


## Step 2: Resolve resource group and principal

In [2]:
# Discover the resource group that contains the existing multi account
MULTI_RG = subprocess.run(
    f'az cognitiveservices account list --query "[?name==\'{MULTI_ACCOUNT}\'].resourceGroup" -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()

assert MULTI_RG, f'Could not find resource group for account "{MULTI_ACCOUNT}" - is az login done?'

# Get subscription ID
SUB_ID = subprocess.run(
    'az account show --query id -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()

# Get principal ID from cached JWT: avoids a network call to graph.microsoft.com
token   = subprocess.run(
    'az account get-access-token --query accessToken -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()
padding = '=' * (4 - len(token.split('.')[1]) % 4)
PRINCIPAL_ID = json.loads(base64.b64decode(token.split('.')[1] + padding))['oid']

# Derive APIM service name from GATEWAY_URL for Step 5
# e.g. https://apim-foundry-{suffix}.azure-api.net/openai -> apim-foundry-{suffix}
APIM_NAME = GATEWAY_URL.split('//')[1].split('.')[0]
SUFFIX    = APIM_NAME.split('-')[-1]            # hub suffix (e.g. {suffix})
CORE_RG    = f'rg-foundry-core-{SUFFIX}'

print(f'Multi account : {MULTI_ACCOUNT}')
print(f'Multi RG      : {MULTI_RG}')
print(f'Subscription  : {SUB_ID}')
print(f'Principal ID  : {PRINCIPAL_ID}')
print(f'Core RG        : {CORE_RG}')
print(f'APIM service  : {APIM_NAME}')

Multi account : aif-spoke-multi-c2676f
Multi RG      : rg-foundry-multi-c2676f
Subscription  : 00000000-0000-0000-0000-000000000000
Principal ID  : 00000000-0000-0000-0000-000000000000
Core RG        : rg-foundry-core-c2676f
APIM service  : apim-foundry-c2676f


## Step 3: Deploy IQ spoke Bicep

Deploys `main.bicep` into the existing `rg-foundry-multi-{suffix}` resource group.
Adds the `iq-search-{suffix}` service and `iq-project` to the shared account.
Takes ~3-5 minutes.

In [3]:
result = subprocess.run(
    [
        'az', 'deployment', 'group', 'create',
        '-g', MULTI_RG,
        '--template-file', 'main.bicep',
        '-p', f'deployerPrincipalId={PRINCIPAL_ID}',
        '-p', f'apimUrl={GATEWAY_URL}',
        '-p', f'apimSubscriptionKey={BOOTSTRAP_KEY}',
        '-p', f'gatewayModelName={CHAT_MODEL}',
        '-p', f'existingAccountName={MULTI_ACCOUNT}',
        # eastus2 was out of Search capacity (InsufficientResourcesAvailable): pin Search to Sweden Central.
        # Other resources still land in the RG region. Change/remove this line if eastus2 has capacity again.
        '-p', f'searchLocation=swedencentral',
        '--name', 'iq-spoke',
        '-o', 'table',
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Bicep deployment failed - see stderr above')

Name      State      Timestamp                         Mode         ResourceGroup
--------  ---------  --------------------------------  -----------  -----------------------
iq-spoke  Succeeded  2026-05-11T19:06:35.962878+00:00  Incremental  rg-foundry-multi-c2676f



## Step 4: Read outputs and write to `.env`

In [4]:
r = subprocess.run(
    f'az deployment group show -g "{MULTI_RG}" -n iq-spoke --query properties.outputs -o json',
    shell=True, capture_output=True, text=True
)
if r.returncode != 0 or not r.stdout.strip():
    raise RuntimeError(f'Failed to read deployment outputs.\n{r.stderr}')

out = json.loads(r.stdout)

PROJECT_NAME     = out['projectName']['value']
PROJECT_ENDPOINT = out['projectEndpoint']['value']
PROJECT_MI       = out['projectManagedIdentityId']['value']
APIM_CONNECTION  = out['apimConnectionName']['value']
SEARCH_ENDPOINT  = out['searchEndpoint']['value']
SEARCH_NAME      = out['searchName']['value']

print(f'Account (existing) : {MULTI_ACCOUNT}')
print(f'Project name       : {PROJECT_NAME}')
print(f'Project endpoint   : {PROJECT_ENDPOINT}')
print(f'APIM connection    : {APIM_CONNECTION}')
print(f'Search endpoint    : {SEARCH_ENDPOINT}')
print(f'Search name        : {SEARCH_NAME}')

# Merge into .env
existing = {}
for line in env_file.read_text().splitlines():
    if '=' in line and not line.startswith('#'):
        k, _, v = line.partition('=')
        existing[k.strip()] = v.strip()

existing.update({
    'IQ_FOUNDRY_PROJECT':          PROJECT_NAME,
    'IQ_FOUNDRY_PROJECT_ENDPOINT': PROJECT_ENDPOINT,
    'IQ_APIM_CONNECTION':          APIM_CONNECTION,
    'IQ_SEARCH_ENDPOINT':          SEARCH_ENDPOINT,
    'IQ_SEARCH_NAME':              SEARCH_NAME,
    'IQ_RESOURCE_GROUP':           MULTI_RG,
    'AZURE_SUBSCRIPTION_ID':       SUB_ID,
})

env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f'\n.env updated: {env_file}')

Account (existing) : aif-spoke-multi-c2676f
Project name       : iq-project
Project endpoint   : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/iq-project
APIM connection    : iq-apim-connection
Search endpoint    : https://iq-search-n5d3ja.search.windows.net
Search name        : iq-search-n5d3ja

.env updated: <repo-root>/.env


## Step 5: Create dedicated IQ APIM subscription

Creates a `foundry-gateway-iq` subscription on the core APIM service, scoped to the
OpenAI API. This gives the IQ workload its own rate-limit bucket - important because
the embedding batch in `10-02-index-and-ingest.ipynb` generates ~6M tokens across 3,000
documents and would otherwise consume Team Alpha's quota.

The dedicated key replaces the bootstrap key on the `iq-apim-connection` connection.

> **If this step fails** (e.g. insufficient permissions on the core APIM), run the
> fallback cell to use `ALPHA_GATEWAY_KEY` instead.

In [5]:
APIM_SUB_NAME = 'foundry-gateway-iq'
APIM_BASE_URI = (
    f'https://management.azure.com/subscriptions/{SUB_ID}'
    f'/resourceGroups/{CORE_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}'
)

create_result = subprocess.run(
    [
        'az', 'rest', '--method', 'PUT',
        '--uri', f'{APIM_BASE_URI}/subscriptions/{APIM_SUB_NAME}?api-version=2024-06-01-preview',
        '--body', json.dumps({
            'properties': {
                'displayName': 'Foundry IQ Gateway Access',
                'scope': (
                    f'/subscriptions/{SUB_ID}/resourceGroups/{CORE_RG}'
                    f'/providers/Microsoft.ApiManagement/service/{APIM_NAME}/apis/openai'
                ),
                'state': 'active',
            }
        }),
        '--headers', 'Content-Type=application/json',
    ],
    capture_output=True, text=True
)

if create_result.returncode != 0:
    print(f'APIM subscription creation failed:\n{create_result.stderr.strip()}')
    print('\nRun the fallback cell below to use ALPHA_GATEWAY_KEY instead.')
    IQ_GATEWAY_KEY = None
else:
    IQ_GATEWAY_KEY = subprocess.run(
        f'az rest --method POST'
        f' --uri "{APIM_BASE_URI}/subscriptions/{APIM_SUB_NAME}/listSecrets?api-version=2024-06-01-preview"'
        f' --query primaryKey -o tsv',
        shell=True, capture_output=True, text=True
    ).stdout.strip()

    # PATCH the landing-zone-apim connection with the dedicated key
    models_list = [{
        'name': CHAT_MODEL,
        'properties': {'model': {'name': CHAT_MODEL, 'version': '', 'format': 'OpenAI'}},
    }]
    connection_uri = (
        f'https://management.azure.com/subscriptions/{SUB_ID}'
        f'/resourceGroups/{MULTI_RG}/providers/Microsoft.CognitiveServices/accounts/{MULTI_ACCOUNT}'
        f'/projects/{PROJECT_NAME}/connections/{APIM_CONNECTION}?api-version=2025-04-01-preview'
    )
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump({
            'properties': {
                'category': 'ApiManagement',
                'target': GATEWAY_URL,
                'authType': 'ApiKey',
                'credentials': {'key': IQ_GATEWAY_KEY},
                'metadata': {
                    'deploymentInPath': 'true',
                    'inferenceAPIVersion': '2024-10-21',
                    'models': json.dumps(models_list),
                },
            }
        }, f)
        payload_file = f.name

    patch = subprocess.run(
        f'az rest --method PATCH --uri "{connection_uri}"'
        f' --body @"{payload_file}" --headers "Content-Type=application/json" -o none',
        shell=True, capture_output=True, text=True
    )
    if patch.returncode == 0:
        print(f'APIM subscription created : {APIM_SUB_NAME}')
        print(f'IQ gateway key            : {IQ_GATEWAY_KEY[:4]}... (hidden)')
        print(f'Connection patched        : {APIM_CONNECTION} on {PROJECT_NAME}')
    else:
        print(f'Connection PATCH failed: {patch.stderr.strip()}')
        IQ_GATEWAY_KEY = None

APIM subscription created : foundry-gateway-iq
IQ gateway key            : 552f... (hidden)
Connection patched        : iq-apim-connection on iq-project


In [6]:
# Fallback: use ALPHA_GATEWAY_KEY if Step 5 failed.
# Uncomment and run only if needed.

# IQ_GATEWAY_KEY = os.environ['ALPHA_GATEWAY_KEY']
# print(f'Using ALPHA_GATEWAY_KEY as IQ_GATEWAY_KEY: {IQ_GATEWAY_KEY[:4]}... (hidden)')

In [7]:
assert IQ_GATEWAY_KEY, 'IQ_GATEWAY_KEY is not set - check Step 5 or run the fallback cell.'

existing = {}
for line in env_file.read_text().splitlines():
    if '=' in line and not line.startswith('#'):
        k, _, v = line.partition('=')
        existing[k.strip()] = v.strip()

existing['IQ_GATEWAY_KEY'] = IQ_GATEWAY_KEY
env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f'IQ_GATEWAY_KEY written to {env_file}')

IQ_GATEWAY_KEY written to <repo-root>/.env


## Step 6: Wait for RBAC propagation

Azure role assignments can take up to 2 minutes to propagate. Running the subsequent
notebooks immediately may produce 403 errors on the search service or project.

In [8]:
from IPython.display import clear_output

for remaining in range(90, 0, -10):
    clear_output(wait=True)
    print(f'Waiting for RBAC to propagate... {remaining}s')
    time.sleep(10)

clear_output(wait=True)
print('RBAC propagation wait complete.')

RBAC propagation wait complete.


## Step 7: Verify deployment

In [9]:
from azure.identity import DefaultAzureCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()

# Verify AI Search
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
indexes = [i.name for i in index_client.list_indexes()]
print(f'Search service : {SEARCH_ENDPOINT}')
print(f'Indexes        : {indexes if indexes else "(none - run 10-02-index-and-ingest.ipynb next)"}')

# Verify Foundry project
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
connections = list(project_client.connections.list())
print(f'\nProject        : {PROJECT_ENDPOINT}')
print(f'Connections:')
for c in connections:
    d = dict(c)
    print(f'  {d.get("name")} ({d.get("type")}) -> {d.get("target")}')

Search service : https://iq-search-n5d3ja.search.windows.net
Indexes        : (none — run 09-02-index-and-ingest.ipynb next)

Project        : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/iq-project
Connections:
  appinsights-connection (AppInsights) -> /subscriptions/00000000-0000-0000-0000-000000000000/resourceGroups/rg-foundry-multi-c2676f/providers/Microsoft.Insights/components/appi-obs-n5d3ja
  iq-apim-connection (ApiManagement) -> https://apim-foundry-c2676f.azure-api.net/openai


## Done

The IQ resources are deployed and `.env` has been updated.

**Keys written to `.env`:**

| Key | Description |
|-----|-------------|
| `IQ_FOUNDRY_PROJECT` | Project name (`iq-project`) |
| `IQ_FOUNDRY_PROJECT_ENDPOINT` | Project endpoint URL |
| `IQ_APIM_CONNECTION` | APIM connection name on `iq-project` |
| `IQ_SEARCH_ENDPOINT` | Azure AI Search service URL |
| `IQ_SEARCH_NAME` | Search service resource name |
| `IQ_GATEWAY_KEY` | Dedicated APIM subscription key for IQ workload |
| `IQ_RESOURCE_GROUP` | Resource group (same as `rg-foundry-multi-{suffix}`) |
| `AZURE_SUBSCRIPTION_ID` | Azure subscription ID |

**Next step:** run `10-02-index-and-ingest.ipynb` to create the `arxiv-nlp` search index and upload documents.

---
## Troubleshooting

### IfMatchPreconditionFailed on redeploy

If you rerun Step 3 after Step 5 has already patched the APIM connection, ARM's
incremental mode will fail with an ETag mismatch. Delete the connection first, then
rerun Step 3.

In [10]:
# Run only if redeployment fails with IfMatchPreconditionFailed, then re-run Step 3.

# connection_uri = (
#     f'https://management.azure.com/subscriptions/{SUB_ID}'
#     f'/resourceGroups/{MULTI_RG}/providers/Microsoft.CognitiveServices/accounts/{MULTI_ACCOUNT}'
#     f'/projects/{PROJECT_NAME}/connections/{APIM_CONNECTION}?api-version=2025-04-01-preview'
# )
# r = subprocess.run(
#     f'az rest --method DELETE --uri "{connection_uri}"',
#     shell=True, capture_output=True, text=True
# )
# print('Deleted' if r.returncode == 0 else r.stderr)